# SITCOM-2089 Evaluate M1M3 hard points individual forces per slew (histogram)

This notebook plots a histogram showing the maximum hardpoint values per slew. See also companion notebook SITCOM-2089_evaluate_hsp_forces_per_slew for more detail.

### Prepare Notebook

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from astropy.time import Time

from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState
from lsst.summit.utils.efdUtils import getEfdData
from lsst_efd_client import EfdClient

import warnings

warnings.filterwarnings("ignore")

In [ ]:
# create a client to retrieve datasets in the EFD database
client = EfdClient("usdf_efd")

### Define functions

#### get previous logged detailedState event from a given time stamp

The status of M1M3 is not persistent in the EFD. In order to get at a given time what is the current status, use this function.

In [ ]:
from lsst.ts.xml.enums.MTM1M3 import DetailedStates


def get_previous_logged_detailedState(df_state, timestamp):
    """
    Get logged detailedState from M1M3 immediately before arbitrary time
    Args:
       df_state (pandas dataframe): pandas dataframe obtained from  time series of
          "lsst.sal.MTM1M3.logevent_detailedState" covering a wide time frame which includes
          the time stamp
       timestamp (pandas timestamp): a timestamp where we want to probe the current status of M1M3
    Returns:
       prev_state: human readable status of current M1M3 status
    """
    df_state_names = df_state["detailedState"].map(lambda x: DetailedStates(x).name)
    previous_index = df_state.index.asof(timestamp)
    try:
        prev = df_state.index.get_loc(previous_index)
    except KeyError:
        return "KeyError"
    return df_state_names[prev]

### Define relevant settings

#### Safety limits

In [ ]:
OPERATIONAL_LIMIT = 500 # newtons
SAFE_LIMIT = 900
BREAKAWAY_LIMIT = 3200

#### Observation day

In [ ]:
## Insert here the day_obs of interest
day_obs = 20250513

### Load data

#### Get slews and tracks

In [ ]:
# Select data from a given date
eventMaker = TMAEventMaker()
events = eventMaker.getEvents(day_obs)

# Get lists of slew and track events
slews = [e for e in events if e.type == TMAState.SLEWING]
tracks = [e for e in events if e.type == TMAState.TRACKING]
print(f"There are {len(events)} events")
print(f"Found {len(slews)} slews and {len(tracks)} tracks")

In [ ]:
df_hp = getEfdData(client, "lsst.sal.MTM1M3.hardpointActuatorData", event=slew)

In [ ]:
# Get slews passing certain criteria
hp_max_hist = np.empty(len(slews))
hp_threshold = OPERATIONAL_LIMIT
df_state = getEfdData(
    client,
    "lsst.sal.MTM1M3.logevent_detailedState",
    begin=Time(slews[0].begin, format="isot", scale="utc"),
    end=Time(slews[-1].end, format="isot", scale="utc"),
)  # get an array for all state changes from first slew of day_obs to final one
for i, slew in enumerate(slews):
    if (
        slew.seqNum == 0
    ):  # skip first one to avoid problems looking for a previous detailedState outside the df_state range
        continue
    df_hp = getEfdData(client, "lsst.sal.MTM1M3.hardpointActuatorData", event=slew)
    timestamp = pd.Timestamp(
        Time(slew.begin, format="iso", scale="utc").value, tz="utc"
    )
    begin_state = get_previous_logged_detailedState(df_state, timestamp)
    if begin_state == "KeyError":
        continue
    timestamp = pd.Timestamp(Time(slew.end, format="iso", scale="utc").value, tz="utc")
    end_state = get_previous_logged_detailedState(df_state, timestamp)
    if len(df_hp) > 0: 
        hp_max_individual = np.array(
            [
                np.max(abs(df_hp["measuredForce0"].values)),
                np.max(abs(df_hp["measuredForce1"].values)),
                np.max(abs(df_hp["measuredForce2"].values)),
                np.max(abs(df_hp["measuredForce3"].values)),
                np.max(abs(df_hp["measuredForce4"].values)),
                np.max(abs(df_hp["measuredForce5"].values)),
            ]
        )
        hp_max = np.max(hp_max_individual)
        hp_max_hist[i] = hp_max

In [ ]:
# Plot a distribution of maximum forces
plt.ylabel("Number of slews")
plt.xlabel("Maximum recorded force in any hardpoint per slew (N)")
plt.title(f"Maximum force on hardpoints for {day_obs}")
plt.axvline(x=OPERATIONAL_LIMIT, color='orange', linestyle='--', linewidth=2, label='Operational limit')
plt.axvline(x=SAFE_LIMIT, color='red', linestyle='--', linewidth=2, label='Safe limit')
plt.legend()
h = plt.hist(hp_max_hist, bins=100, range=[0, 2000])